In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import chi2_contingency, fisher_exact


1.Load data & Data Overview

In [27]:
df = pd.read_excel("../data/2022Cdata.xlsx")

In [28]:
print(df.shape)
df.info()
df.head(10)

(58, 5)
<class 'pandas.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   文物编号    58 non-null     int64
 1   纹饰      58 non-null     str  
 2   类型      58 non-null     str  
 3   颜色      54 non-null     str  
 4   表面风化    58 non-null     str  
dtypes: int64(1), str(4)
memory usage: 2.4 KB


,文物编号,纹饰,类型,颜色,表面风化
0,1,C,高钾,蓝绿,无风化
1,2,A,铅钡,浅蓝,风化
2,3,A,高钾,蓝绿,无风化
3,4,A,高钾,蓝绿,无风化
4,5,A,高钾,蓝绿,无风化
5,6,A,高钾,蓝绿,无风化
6,7,B,高钾,蓝绿,风化
7,8,C,铅钡,紫,风化
8,9,B,高钾,蓝绿,风化
9,10,B,高钾,蓝绿,风化


2. Missing Value

In [29]:
df.isnull().sum()

文物编号    0
纹饰      0
类型      0
颜色      4
表面风化    0
dtype: int64

3. Value counts

In [30]:
for col in ["类型","纹饰","颜色","表面风化"]:

        print("="*20)
        print(col)
        print(df[col].value_counts(dropna=False))
        

类型
类型
铅钡    40
高钾    18
Name: count, dtype: int64
纹饰
纹饰
C    30
A    22
B     6
Name: count, dtype: int64
颜色
颜色
浅蓝     20
蓝绿     15
深绿      7
紫       4
NaN     4
浅绿      3
深蓝      2
黑       2
绿       1
Name: count, dtype: int64
表面风化
表面风化
风化     34
无风化    24
Name: count, dtype: int64


In [31]:
print(df.columns)

Index(['文物编号', '纹饰', '类型', '颜色', '表面风化'], dtype='str')


pre data cleaning

In [ ]:
df_2 = pd.read_excel("../data/2022Cdata.xlsx", sheet_name="表单2")
df_2.info()


<class 'pandas.DataFrame'>
RangeIndex: 69 entries, 0 to 68
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   文物采样点        69 non-null     str    
 1   二氧化硅(SiO2)   69 non-null     float64
 2   氧化钠(Na2O)    19 non-null     float64
 3   氧化钾(K2O)     41 non-null     float64
 4   氧化钙(CaO)     61 non-null     float64
 5   氧化镁(MgO)     44 non-null     float64
 6   氧化铝(Al2O3)   68 non-null     float64
 7   氧化铁(Fe2O3)   45 non-null     float64
 8   氧化铜(CuO)     64 non-null     float64
 9   氧化铅(PbO)     58 non-null     float64
 10  氧化钡(BaO)     53 non-null     float64
 11  五氧化二磷(P2O5)  59 non-null     float64
 12  氧化锶(SrO)     46 non-null     float64
 13  氧化锡(SnO2)    7 non-null      float64
 14  二氧化硫(SO2)    8 non-null      float64
dtypes: float64(14), str(1)
memory usage: 8.2 KB


In [38]:
print(df_2.shape)
df_2.head()

(69, 15)


,文物采样点,二氧化硅(SiO2),氧化钠(Na2O),氧化钾(K2O),氧化钙(CaO),氧化镁(MgO),氧化铝(Al2O3),氧化铁(Fe2O3),氧化铜(CuO),氧化铅(PbO),氧化钡(BaO),五氧化二磷(P2O5),氧化锶(SrO),氧化锡(SnO2),二氧化硫(SO2)
0,01,69.33,NaN,9.99,6.32,0.87,3.93,1.74,3.87,NaN,NaN,1.17,NaN,NaN,0.39
1,02,36.28,NaN,1.05,2.34,1.18,5.73,1.86,0.26,47.43,NaN,3.57,0.19,NaN,NaN
2,03部位1,87.05,NaN,5.19,2.01,NaN,4.06,NaN,0.78,0.25,NaN,0.66,NaN,NaN,NaN
3,03部位2,61.71,NaN,12.37,5.87,1.11,5.50,2.16,5.09,1.41,2.86,0.70,0.10,NaN,NaN
4,04,65.88,NaN,9.67,7.12,1.56,6.44,2.06,2.18,NaN,NaN,0.79,NaN,NaN,0.36


In [37]:
df_2.isnull().sum()

文物采样点           0
二氧化硅(SiO2)      0
氧化钠(Na2O)      50
氧化钾(K2O)       28
氧化钙(CaO)        8
氧化镁(MgO)       25
氧化铝(Al2O3)      1
氧化铁(Fe2O3)     24
氧化铜(CuO)        5
氧化铅(PbO)       11
氧化钡(BaO)       16
五氧化二磷(P2O5)    10
氧化锶(SrO)       23
氧化锡(SnO2)      62
二氧化硫(SO2)      61
dtype: int64

In [39]:
component_cols = df_2.columns[1:]
component_cols

Index(['二氧化硅(SiO2)', '氧化钠(Na2O)', '氧化钾(K2O)', '氧化钙(CaO)', '氧化镁(MgO)',
       '氧化铝(Al2O3)', '氧化铁(Fe2O3)', '氧化铜(CuO)', '氧化铅(PbO)', '氧化钡(BaO)',
       '五氧化二磷(P2O5)', '氧化锶(SrO)', '氧化锡(SnO2)', '二氧化硫(SO2)'],
      dtype='str')

In [40]:
df_2["成分总和"] = df_2[component_cols].fillna(0).sum(axis=1)
df_2[["文物采样点", "成分总和"]].head(10)

,文物采样点,成分总和
0,01,97.61
1,02,99.89
2,03部位1,100.00
3,03部位2,98.88
4,04,96.06
5,05,96.51
6,06部位1,98.92
7,06部位2,98.84
8,07,99.70
9,08,99.82


In [43]:
df_2["是否有效"] = df_2["成分总和"]. between (85, 105)
df_2["是否有效"].value_counts(dropna=False)

是否有效
True     67
False     2
Name: count, dtype: int64

In [45]:
df_2_valid = df_2[df_2["是否有效"]].copy()
df_2_valid.head()

,文物采样点,二氧化硅(SiO2),氧化钠(Na2O),氧化钾(K2O),氧化钙(CaO),氧化镁(MgO),氧化铝(Al2O3),氧化铁(Fe2O3),氧化铜(CuO),氧化铅(PbO),氧化钡(BaO),五氧化二磷(P2O5),氧化锶(SrO),氧化锡(SnO2),二氧化硫(SO2),成分总和,是否有效
0,01,69.33,NaN,9.99,6.32,0.87,3.93,1.74,3.87,NaN,NaN,1.17,NaN,NaN,0.39,97.61,True
1,02,36.28,NaN,1.05,2.34,1.18,5.73,1.86,0.26,47.43,NaN,3.57,0.19,NaN,NaN,99.89,True
2,03部位1,87.05,NaN,5.19,2.01,NaN,4.06,NaN,0.78,0.25,NaN,0.66,NaN,NaN,NaN,100.00,True
3,03部位2,61.71,NaN,12.37,5.87,1.11,5.50,2.16,5.09,1.41,2.86,0.70,0.10,NaN,NaN,98.88,True
4,04,65.88,NaN,9.67,7.12,1.56,6.44,2.06,2.18,NaN,NaN,0.79,NaN,NaN,0.36,96.06,True


## 1. Cross-tabulation between Glass Type and Weathering

Purpose:
To summarize the frequency distribution of weathering status under different glass types.

In [ ]:
table_type_display=pd.crosstab( 
    df["类型"], df["表面风化"] , margins=True)
table_type

表面风化,无风化,风化
类型,,
铅钡,12,28
高钾,12,6


In [ ]:
table_type=pd.crosstab( 
    df["类型"], df["表面风化"] )
table_type

表面风化,无风化,风化
类型,,
铅钡,12,28
高钾,12,6


### Chi-square Test

In [ ]:
result=chi2_contingency(
    table_type, 
    correction=False)
result


Chi2ContingencyResult(statistic=np.float64(6.8803921568627455), pvalue=np.float64(0.008714644061182652), dof=1, expected_freq=array([[16.55172414, 23.44827586],
       [ 7.44827586, 10.55172414]]))

In [ ]:
print("Chi-square stastistic :", result.statistic )
print("p-value:",result.pvalue)
print( "degree of freedom:" , result.dof)
print("expected frequency:",result.expected_freq)

Chi-square stastistic : 6.8803921568627455
p-value: 0.008714644061182652
degree of freedom: 1
expected frequency: [[16.55172414 23.44827586]
 [ 7.44827586 10.55172414]]


### expected cross tab under H0

In [ ]:
expected = pd.DataFrame(
    result.expected_freq,
    index=table_type.index,
    columns=table_type.columns
)

expected

表面风化,无风化,风化
类型,,
铅钡,16.551724,23.448276
高钾,7.448276,10.551724


In [ ]:
print("Minimum expected frequency:", expected.min().min())


Minimum expected frequency: 7.448275862068965


### Chi-square Assumption Check

All expected frequencies are greater than 5.

Therefore, the assumptions of the Pearson Chi-square test are satisfied.

Since p-value < 0.05,

the null hypothesis of independence is rejected.

Therefore,

glass type and weathering status are significantly associated.

## Cramer's V Test

In [ ]:
type_prop = pd.crosstab (df["类型"], df["表面风化"], normalize="index")
type_prop

表面风化,无风化,风化
类型,,
铅钡,0.300000,0.700000
高钾,0.666667,0.333333


In [ ]:
n = table_type.to_numpy().sum()
r,c = table_type.shape

cramers_v = np.sqrt(result.statistic /(n * min(r-1,c-1)))
cramers_v

np.float64(0.3444233600968322)

fisher exact test

In [ ]:
from scipy.stats import fisher_exact
fisher_result = fisher_exact(table_type)
fisher_result

SignificanceResult(statistic=np.float64(0.21428571428571427), pvalue=np.float64(0.011327604477116615))

# Relationship 2

## Decoration vs weathering

### Cross Table

In [ ]:
table_decoration = pd.crosstab (df["纹饰"], df["表面风化"])
table_decoration

表面风化,无风化,风化
纹饰,,
A,11,11
B,0,6
C,13,17


### Chi-square Tset

In [ ]:
decoration_result = chi2_contingency (table_decoration, correction=False)
print(decoration_result.pvalue)
print(decoration_result.expected_freq)

0.08388839673210008
[[ 9.10344828 12.89655172]
 [ 2.48275862  3.51724138]
 [12.4137931  17.5862069 ]]


Since %33 of the data < 5 , chi-square may not be accurate.

So we try Monte Carlo chi-square.

### Monte Carlo Chi-square

In [ ]:
from scipy.stats import chi2_contingency, MonteCarloMethod


In [ ]:
mc_method = MonteCarloMethod(n_resamples=100000)

decoration_mc_result = chi2_contingency(table_decoration, correction=False, method=mc_method)
decoration_mc_result

Chi2ContingencyResult(statistic=np.float64(4.9565359477124185), pvalue=np.float64(0.10386896131038689), dof=nan, expected_freq=array([[ 9.10344828, 12.89655172],
       [ 2.48275862,  3.51724138],
       [12.4137931 , 17.5862069 ]]))

In [ ]:
print ("Monte Carlo p-value:", decoration_mc_result.pvalue)

Monte Carlo p-value: 0.10386896131038689


p-value > 0.05 , decoration and weathering are independent and unrelated.

### Cramers' V

In [ ]:
n = table_decoration.to_numpy().sum()
r,c  = table_decoration.shape
decoration_cramers_v = np.sqrt( decoration_result.statistic/(n*min(n-1,c-1)))
decoration_cramers_v



np.float64(0.29233117579189066)

“Cramér’s V 为 0.29，表明样本中存在弱至接近中等程度的关联趋势。然而 Monte Carlo 检验 p\approx0.10>0.05，故该关联未达到统计显著水平。”
对纹饰与表面风化建立 3\times2 列联表后发现，部分单元格理论频数小于5，普通 Pearson 卡方检验的渐近近似条件不够理想，因此进一步采用 Monte Carlo 方法进行显著性检验。Monte Carlo 检验得到 p\approx0.10>0.05，故在5%的显著性水平下，尚无充分证据认为纹饰与表面风化之间存在显著统计关联。另一方面，Cramér’s V 约为0.29，表明样本数据中仍呈现一定程度的关联趋势，其中B类纹饰的6件文物均发生风化；但由于B类样本量较小，该现象的稳定性仍需更多样本验证。因此，本文不将纹饰视为表面风化的显著相关因素。

## Relationship between color & weathering
by using category merging

In [ ]:
df["颜色"].value_counts(dropna=False)

颜色
浅蓝     20
蓝绿     15
深绿      7
紫       4
NaN     4
浅绿      3
深蓝      2
黑       2
绿       1
Name: count, dtype: int64

In [ ]:
df[df["颜色"].isna()]

,文物编号,纹饰,类型,颜色,表面风化
18,19,A,铅钡,NaN,风化
39,40,C,铅钡,NaN,风化
47,48,A,铅钡,NaN,风化
57,58,C,铅钡,NaN,风化


颜色变量存在4条缺失记录，且均对应风化文物，因此颜色变量缺失并非完全随机。后续颜色相关分析仅基于颜色信息完整的数据进行，并将该缺失情况作为研究局限之一。

### 删除颜色缺失的数据

In [ ]:
df_color = df.dropna(subset=["颜色"])


In [ ]:
color_table = pd.crosstab( df_color["颜色"] , df_color["表面风化"])
color_table

表面风化,无风化,风化
颜色,,
浅绿,2,1
浅蓝,8,12
深绿,3,4
深蓝,2,0
紫,2,2
绿,1,0
蓝绿,6,9
黑,0,2


### category merging

In [ ]:
color_result = chi2_contingency(color_table, correction=False)
df_expected_color = pd.DataFrame (color_result.expected_freq, index=color_table.index, columns=color_table.columns)
df_expected_color

表面风化,无风化,风化
颜色,,
浅绿,1.333333,1.666667
浅蓝,8.888889,11.111111
深绿,3.111111,3.888889
深蓝,0.888889,1.111111
紫,1.777778,2.222222
绿,0.444444,0.555556
蓝绿,6.666667,8.333333
黑,0.888889,1.111111


In [ ]:
color_mc_result = chi2_contingency(color_table, correction=False, method=mc_method)
print("Monte Carlo p-value of color:", color_mc_result.pvalue)

Monte Carlo p-value of color: 0.5796942030579694


start merging

In [ ]:
df_color_merge = df_color.copy()
df_color_merge["颜色"] = df_color_merge["颜色"].replace({"浅蓝":"蓝系",
                                                     "深蓝":"蓝系",
                                                     "蓝绿":"蓝系",
                                                     "浅绿":"绿系",
                                                     "深绿":"绿系",
                                                     "绿":"绿系"})
df_color_merge["颜色"].value_counts()

颜色
蓝系    37
绿系    11
紫      4
黑      2
Name: count, dtype: int64

In [ ]:
color_table_merge = pd.crosstab(df_color_merge["颜色"], df_color_merge["表面风化"])
color_table_merge

表面风化,无风化,风化
颜色,,
紫,2,2
绿系,6,5
蓝系,16,21
黑,0,2


In [ ]:
color_result_merge = chi2_contingency(color_table_merge, correction=False)
color_table_merge_expected = pd.DataFrame( color_result_merge.expected_freq, index=color_table_merge.index, columns=color_table_merge.columns)
color_table_merge_expected

表面风化,无风化,风化
颜色,,
紫,1.777778,2.222222
绿系,4.888889,6.111111
蓝系,16.444444,20.555556
黑,0.888889,1.111111


In [ ]:
color_merge_mc_result = chi2_contingency (color_table_merge, correction=False, method=mc_method)

In [ ]:
print(color_merge_mc_result.pvalue)

0.6552734472655274


In [ ]:
n_color = color_table_merge.to_numpy().sum()
r_color, c_color = color_table_merge.shape

In [ ]:
cramers_v_color_merge = np.sqrt( color_result_merge.statistic/(n_color * min (r_color-1,c_color-1)))
print("Cramers_V of color_merge & weathering:", cramers_v_color_merge)

Cramers_V of color_merge & weathering: 0.19842747887695483
